# Attention & Transformer Blocks from Scratch

Shape bookkeeping is 80% of what interviewers grade in transformer implementation questions. This note implements scaled dot-product attention, multi-head attention, LayerNorm, and a full encoder block — with dimension annotations on every single line.

## What Interviewers Test
- Shape tracking through the QKV projection and reshape/transpose dance
- Masking: causal (autoregressive) vs padding masks — what they prevent and how they're applied
- LayerNorm vs BatchNorm — why transformers use LayerNorm
- Positional encoding: sinusoidal construction and intuition
- Multi-head attention: why split into heads, then concatenate
- Numerical stability in attention softmax

## Notation & Dimensions

| Symbol | Meaning | Typical value |
|---|---|---|
| B | Batch size | 8–64 |
| T | Sequence length (tokens) | 128–8192 |
| d_model | Model dimension | 512–4096 |
| H | Number of heads | 8–32 |
| d_k = d_model/H | Per-head dimension | 64 |
| d_ff | Feed-forward inner dim | 4 × d_model |


## Scaled Dot-Product Attention (NumPy)

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

The $1/\sqrt{d_k}$ scaling prevents the dot products from growing large when $d_k$ is large, which would push softmax into near-zero gradient regions.

> 💡 **Interview Tip:** Interviewers always ask: *"Why scale by $\sqrt{d_k}$?"* Answer: dot products of random unit vectors have variance $d_k$; dividing by $\sqrt{d_k}$ normalizes variance to 1, keeping softmax in a responsive range.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.random.seed(42)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)  # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (B, T_q, d_k)
    K: (B, T_k, d_k)
    V: (B, T_k, d_v)
    mask: (B, 1, T_q, T_k) or (B, 1, 1, T_k) — True positions are masked (set to -inf)
    Returns: (B, T_q, d_v)
    """
    B, T_q, d_k = Q.shape
    T_k = K.shape[1]

    # Compute attention scores
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)  # (B, T_q, T_k)

    # Apply mask (fill masked positions with -inf before softmax)
    if mask is not None:
        scores = scores + mask * -1e9   # mask=True → -inf → softmax→0

    attn_weights = softmax(scores, axis=-1)  # (B, T_q, T_k)
    output = attn_weights @ V                # (B, T_q, d_v)
    return output, attn_weights

# Quick shape check
B, T, d_k, d_v = 2, 6, 16, 16
Q = np.random.randn(B, T, d_k)
K = np.random.randn(B, T, d_k)
V = np.random.randn(B, T, d_v)
out, weights = scaled_dot_product_attention(Q, K, V)
print(f"Q shape: {Q.shape}  ->  output: {out.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Weights sum to 1 per row: {np.allclose(weights.sum(axis=-1), 1.0)}")


## Causal Mask vs Padding Mask

| | Causal (autoregressive) | Padding |
|---|---|---|
| **Purpose** | Prevent attending to future tokens | Prevent attending to `[PAD]` tokens |
| **Shape** | (T, T) lower-triangular | (B, 1, 1, T_k) — per-sequence |
| **Used in** | Decoder self-attention | Encoder self-attention, cross-attention |
| **Applied** | Once per model, reused for all heads | Per batch, different per example |

> 💡 **Interview Tip:** A common gotcha — the causal mask is on the *query* side (rows), not the key side. Row $i$ can attend to keys $0..i$ only.


In [ ]:
def make_causal_mask(T):
    """Lower-triangular mask. Position (i,j) is True (masked) when j > i."""
    mask = np.triu(np.ones((T, T), dtype=bool), k=1)  # upper triangle = True = masked
    return mask[np.newaxis, np.newaxis, :, :]           # (1, 1, T, T) for broadcasting

def make_padding_mask(lengths, max_len):
    """
    lengths: (B,) array of actual sequence lengths
    Returns: (B, 1, 1, max_len) bool mask where True = padding = should be masked
    """
    B = len(lengths)
    mask = np.arange(max_len)[np.newaxis, :] >= lengths[:, np.newaxis]  # (B, max_len)
    return mask[:, np.newaxis, np.newaxis, :]                             # (B, 1, 1, max_len)

T = 5
causal = make_causal_mask(T).squeeze()
print("Causal mask (True=masked):")
print(causal.astype(int))

lengths = np.array([3, 5, 4])
pad_mask = make_padding_mask(lengths, max_len=5).squeeze()
print("\nPadding mask for lengths [3,5,4] (True=masked):")
print(pad_mask.astype(int))


## Multi-Head Attention

The reshape/transpose dance is the hardest part to get right. Every line needs a shape comment.


In [ ]:
class MultiHeadAttention:
    """NumPy multi-head attention. All weights are (d_model, d_model)."""

    def __init__(self, d_model, n_heads):
        assert d_model % n_heads == 0
        self.d_model  = d_model   # total model dim
        self.n_heads  = n_heads   # H
        self.d_k      = d_model // n_heads  # per-head dim

        # Projection matrices (d_model, d_model)
        scale = np.sqrt(2 / d_model)
        self.W_Q = np.random.randn(d_model, d_model) * scale
        self.W_K = np.random.randn(d_model, d_model) * scale
        self.W_V = np.random.randn(d_model, d_model) * scale
        self.W_O = np.random.randn(d_model, d_model) * scale

    def forward(self, Q_in, K_in, V_in, mask=None):
        B, T_q, _ = Q_in.shape   # (B, T_q, d_model)
        T_k       = K_in.shape[1]

        # ---- Linear projections ----
        Q = Q_in @ self.W_Q   # (B, T_q, d_model)
        K = K_in @ self.W_K   # (B, T_k, d_model)
        V = V_in @ self.W_V   # (B, T_k, d_model)

        # ---- Split into H heads ----
        # Reshape: (B, T, d_model) → (B, T, H, d_k) → (B, H, T, d_k)
        def split_heads(x, T):
            x = x.reshape(B, T, self.n_heads, self.d_k)   # (B, T, H, d_k)
            return x.transpose(0, 2, 1, 3)                 # (B, H, T, d_k)

        Q = split_heads(Q, T_q)   # (B, H, T_q, d_k)
        K = split_heads(K, T_k)   # (B, H, T_k, d_k)
        V = split_heads(V, T_k)   # (B, H, T_k, d_k)

        # ---- Scaled dot-product attention per head ----
        # Flatten B and H for batch attention: (B*H, T, d_k)
        BH = B * self.n_heads
        Q_flat = Q.reshape(BH, T_q, self.d_k)
        K_flat = K.reshape(BH, T_k, self.d_k)
        V_flat = V.reshape(BH, T_k, self.d_k)

        ctx, attn = scaled_dot_product_attention(Q_flat, K_flat, V_flat, mask=mask)
        # ctx: (B*H, T_q, d_k)

        # ---- Merge heads ----
        ctx = ctx.reshape(B, self.n_heads, T_q, self.d_k)   # (B, H, T_q, d_k)
        ctx = ctx.transpose(0, 2, 1, 3)                       # (B, T_q, H, d_k)
        ctx = ctx.reshape(B, T_q, self.d_model)               # (B, T_q, d_model)

        # ---- Output projection ----
        output = ctx @ self.W_O   # (B, T_q, d_model)
        return output

# Shape check
B, T, d_model, n_heads = 2, 8, 64, 4
mha = MultiHeadAttention(d_model, n_heads)
x = np.random.randn(B, T, d_model)
out = mha.forward(x, x, x)    # self-attention
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
assert out.shape == (B, T, d_model), "Output shape mismatch!"
print("Shape check passed ✓")


## LayerNorm & Feed-Forward Block


In [ ]:
def layer_norm(x, gamma, beta, eps=1e-6):
    """
    x: (B, T, d_model)
    Normalizes over last dimension (per token, across features).
    """
    mu  = x.mean(axis=-1, keepdims=True)       # (B, T, 1)
    var = x.var(axis=-1, keepdims=True)         # (B, T, 1)
    x_norm = (x - mu) / np.sqrt(var + eps)     # (B, T, d_model)
    return gamma * x_norm + beta               # learnable scale/shift

def feed_forward(x, W1, b1, W2, b2):
    """
    x:  (B, T, d_model)
    W1: (d_model, d_ff)   W2: (d_ff, d_model)
    """
    h = np.maximum(0, x @ W1 + b1)   # (B, T, d_ff) — ReLU (GELU in modern models)
    return h @ W2 + b2                # (B, T, d_model)

class TransformerEncoderBlock:
    """Pre-norm transformer encoder block."""
    def __init__(self, d_model, n_heads, d_ff):
        self.mha     = MultiHeadAttention(d_model, n_heads)
        self.gamma1  = np.ones(d_model);   self.beta1 = np.zeros(d_model)
        self.gamma2  = np.ones(d_model);   self.beta2 = np.zeros(d_model)
        scale = np.sqrt(2 / d_model)
        self.W1 = np.random.randn(d_model, d_ff)    * scale
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model)    * scale
        self.b2 = np.zeros(d_model)

    def forward(self, x, mask=None):
        # Sub-layer 1: Self-attention + residual
        x_norm = layer_norm(x, self.gamma1, self.beta1)   # (B, T, d_model)
        attn_out = self.mha.forward(x_norm, x_norm, x_norm, mask=mask)
        x = x + attn_out                                   # residual connection

        # Sub-layer 2: Feed-forward + residual
        x_norm = layer_norm(x, self.gamma2, self.beta2)   # (B, T, d_model)
        ff_out = feed_forward(x_norm, self.W1, self.b1, self.W2, self.b2)
        x = x + ff_out                                     # residual connection
        return x                                           # (B, T, d_model)

block = TransformerEncoderBlock(d_model=64, n_heads=4, d_ff=256)
x = np.random.randn(2, 8, 64)
out = block.forward(x)
print(f"Encoder block: {x.shape} → {out.shape}")


## Sinusoidal Positional Encoding
$$PE_{pos,2i} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \quad PE_{pos,2i+1} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$


In [ ]:
def sinusoidal_positional_encoding(T, d_model):
    """
    Returns: (1, T, d_model) — ready to add to token embeddings
    """
    pos  = np.arange(T)[:, np.newaxis]           # (T, 1)
    dims = np.arange(0, d_model, 2)[np.newaxis, :]  # (1, d_model/2) even indices
    angles = pos / (10000 ** (dims / d_model))   # (T, d_model/2)

    PE = np.zeros((T, d_model))
    PE[:, 0::2] = np.sin(angles)   # even positions
    PE[:, 1::2] = np.cos(angles)   # odd positions
    return PE[np.newaxis, :, :]    # (1, T, d_model) for batch broadcast

PE = sinusoidal_positional_encoding(T=50, d_model=64)
print(f"Positional encoding shape: {PE.shape}")
# Visualize
fig, ax = plt.subplots(figsize=(8, 3))
im = ax.imshow(PE[0, :20, :32].T, aspect='auto', cmap='RdBu')
ax.set_xlabel('Position'); ax.set_ylabel('Dimension')
ax.set_title('Sinusoidal Positional Encoding (first 20 positions, 32 dims)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('/tmp/pe.png', dpi=80)
plt.close()
print("PE visualization saved")
print(f"PE range: [{PE.min():.2f}, {PE.max():.2f}] — bounded by sin/cos")


## Common Interview Questions

**Q: Why is multi-head attention better than single-head?**
Different heads can attend to different semantic relationships simultaneously — one head might capture syntactic dependencies, another coreference, another local vs long-range patterns. With a single head, all that information must be compressed into one attention pattern. In practice, empirical results consistently show multi-head > single-head for the same parameter count.

**Q: What is the time and space complexity of self-attention?**
$O(T^2 d)$ time and $O(T^2 + Td)$ space per layer — the attention matrix is $T \times T$. This quadratic scaling in sequence length is why attention is expensive for long contexts and motivates sparse attention, linear attention, and FlashAttention.

**Q: Why LayerNorm instead of BatchNorm in transformers?**
BatchNorm normalizes across the batch dimension — it requires a sufficiently large batch and behaves differently between train and inference. LayerNorm normalizes across features per token, which is batch-size independent and consistent between train/test. Variable-length sequences in NLP make BatchNorm additionally awkward.

**Q: Why does the causal mask use -inf (or a large negative)?**
The causal mask sets masked attention logits to -∞ before softmax. After softmax, $e^{-\infty}=0$, so masked positions receive zero attention weight. Using -1e9 (rather than true -inf) avoids NaN when all logits in a row are -inf (all-padding edge case).

**Q: What is the difference between pre-norm and post-norm transformers?**
Post-norm (original): LayerNorm *after* the residual connection. Pre-norm: LayerNorm *before* the sub-layer, then residual added. Pre-norm is more stable to train (especially deep), requires less warmup, and is the standard in modern models (GPT, Llama, etc.). Post-norm can achieve slightly higher final performance with careful tuning.

## Key Takeaways
- Every line of transformer code needs a shape comment — that's what interviewers grade
- Multi-head reshape: `(B, T, d_model) → (B, T, H, d_k) → (B, H, T, d_k)`; reverse after attention
- Scale scores by $1/\sqrt{d_k}$ to keep softmax responsive as dimension grows
- Causal mask: upper-triangular True matrix applied as -∞ additive bias before softmax
- Padding mask: per-batch boolean mask marking pad token positions
- LayerNorm normalizes per-token across features; BatchNorm normalizes across batch — only LayerNorm works for variable-length sequences
- Pre-norm transformers are more stable to train than original post-norm architecture